# Practical-04: Prompt Engineering Experiments

### Setup the System configuration & check AWS config

In [1]:
import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils.config import config
print("Region:", config.region_name)
print("Default model:", config.default_model_id)

Region: us-east-1
Default model: amazon.nova-lite-v1:0


## Zero-Shot vs Few-Shot vs Chain-of-Thought
- **Zero-shot** — task instructions + category list only. Cheapest, fastest, no worked examples.
- **Few-shot** — adds examples before the real ticket, to prime the exact decision boundary we want.
- **Chain-of-thought (CoT)** — solves the complex task which is ambiguous to the problem

In [2]:
from src.variants import compare_variants, CATEGORIES

print("Categories:", CATEGORIES)

Categories: ['Payroll', 'Leave & Attendance', 'IT Support', 'Policy & General']


In [3]:
test_tickets = [
    "I haven't received my reimbursement for last month's travel expenses.",   # clear: Payroll
    "How many casual leaves do I have left this quarter?",                     # clear: Leave & Attendance
    "My work laptop keeps disconnecting from the VPN, please help.",           # clear: IT Support
    # Deliberately ambiguous — mentions both payroll AND leave:
    "I took unpaid leave last week but my salary this month looks wrong, can you check both?",
]

for ticket in test_tickets:

    print("TICKET:", ticket)

    results = compare_variants(ticket)
    for variant_name, res in results.items():
        print(f"\n[{variant_name.upper()}]")
        print(res["text"])
    print()

TICKET: I haven't received my reimbursement for last month's travel expenses.

[ZERO_SHOT]
Policy & General

[FEW_SHOT]
Payroll

[COT]
Reasoning: The ticket mentions "reimbursement for last month's travel expenses," which relates to financial transactions and expense management. This topic is most closely associated with payroll processes, as reimbursements for expenses are typically handled through payroll systems.

Category: Payroll

TICKET: How many casual leaves do I have left this quarter?

[ZERO_SHOT]
Leave & Attendance

[FEW_SHOT]
Leave & Attendance

[COT]
Reasoning: The ticket is asking about the number of casual leaves remaining for the quarter, which pertains to tracking and managing leave balances. This is directly related to the employee's leave entitlements and usage, which falls under the category of Leave & Attendance.

Category: Leave & Attendance

TICKET: My work laptop keeps disconnecting from the VPN, please help.

[ZERO_SHOT]
IT Support

[FEW_SHOT]
IT Support

[COT]